# FastAPI

A friendly FastAPI notebook with clear explanations and small runnable examples for each concept.

## Installation and Setup

Install FastAPI and Uvicorn in a virtual environment. Uvicorn is the server that runs the app.

In [ ]:
# Run in a terminal or notebook cell if supported.
pip install fastapi uvicorn[standard]

## What is FastAPI?

FastAPI is a modern Python web framework for building APIs. It is designed for speed, developer productivity, and clear code. It uses Python type hints to provide automatic validation and interactive documentation.

### Key points

- Fast performance thanks to Starlette and Pydantic.
- Automatic request validation and response serialization.
- Built-in OpenAPI documentation.
- Works with async Python code naturally.

## How FastAPI differs from other frameworks

- **Flask**: Minimal and synchronous by default. FastAPI adds automatic validation, async support, and docs with less configuration.
- **Django**: Full-stack framework with ORM, templating, and admin. FastAPI focuses on APIs and is much lighter for service endpoints.
- **FastAPI**: Best for API-first projects where speed, schema validation, and documentation matter.

## Uvicorn Server

Uvicorn is an ASGI server that runs FastAPI applications. Use `uvicorn` to serve the app locally or in production.

In [ ]:
# Example command to run in a terminal
# uvicorn main:app --reload --port 8000

## Creating First API

Create a simple FastAPI app with a root route and a POST endpoint that accepts JSON.

In [ ]:
app_code = '''from fastapi import FastAPI

app = FastAPI()

@app.get('/')
async def read_root():
    return {'message': 'Hello, FastAPI!'}

@app.post('/items/')
async def create_item(item: dict):
    return {'received': item}
'''

with open('main.py', 'w', encoding='utf-8') as f:
    f.write(app_code)

print('Wrote main.py. Run: uvicorn main:app --reload --port 8000')

## App Instance

The FastAPI application is created from `FastAPI()`. This object is the main app, and you attach routes, middleware, and event handlers to it.

In [ ]:
from fastapi import FastAPI

app = FastAPI(
    title='My API',
    description='A small FastAPI example',
    version='1.0.0',
)


# To route from one url to another url
app.include_router()

# To add middleware
app.add_middleware()

@app.get('/info')
async def info():
    return {'name': 'FastAPI app', 'version': '1.0.0'}


## Dependency - Depends()

Dependencies let you share common parameters and logic between endpoints. Use `Depends()` to declare a dependency function.

In [ ]:
from fastapi import Depends, FastAPI

app = FastAPI()

async def common_parameters(q: str | None = None, limit: int = 10):
    return {'q': q, 'limit': limit}

@app.get('/search')
async def search(params: dict = Depends(common_parameters)):
    return params


## Pydantic

Pydantic models define data schemas, validate input, and generate clean JSON output. FastAPI uses them for request bodies, query parameters, and responses.

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class Item(BaseModel):
    name: str
    description: str | None = None
    price: float
    tax: float | None = None

@app.post('/items/')
async def create_item(item: Item):
    result = item.dict()
    if item.tax:
        result['price_with_tax'] = item.price + item.tax
    return result


## Middleware - FastAPI Middleware

Middleware runs on every request and response. It is useful for logging, authorization, and modifying the request/response objects.

In [ ]:
from fastapi import FastAPI, Request

app = FastAPI()

@app.middleware('http')
async def log_requests(request: Request, call_next):
    print(f'Received: {request.method} {request.url.path}')
    response = await call_next(request)
    print(f'Response status: {response.status_code}')
    return response


## App Structure - FastAPI Folder Structure

A common FastAPI layout keeps code organized by feature. Below is a simple structure for a small project.

Example project layout:

`````
project/
  app/
    main.py
    api/
      routers.py
    models.py
    dependencies.py
    core.py
  tests/
  requirements.txt
`````

- `main.py`: creates `app = FastAPI()` and includes routers.
- `api/routers.py`: defines endpoint routers.
- `models.py`: Pydantic models.
- `dependencies.py`: shared dependency functions.
- `core.py`: configuration and startup events.

## Background Tasks - Background Tasks in FastAPI

Background tasks allow work to happen after the response is sent, without blocking the client. Common uses include sending emails or writing logs.

In [ ]:
from fastapi import BackgroundTasks, FastAPI

app = FastAPI()

def write_log(message: str):
    with open('background.log', 'a', encoding='utf-8') as file:
        file.write(message + '\n')

@app.post('/notify/')
async def notify(background_tasks: BackgroundTasks, email: str):
    background_tasks.add_task(write_log, f'Notify {email}')
    return {'status': 'notification queued'}


## Async Programming

FastAPI supports async endpoints using `async def`. This is ideal for I/O-bound operations like database access, HTTP requests, and file operations.

In [ ]:
import asyncio
from fastapi import FastAPI

app = FastAPI()

@app.get('/wait/')
async def wait_endpoint():
    await asyncio.sleep(1)
    return {'status': 'finished', 'delay': 1}


## API documentation

FastAPI generates documentation automatically from your routes, type hints, and Pydantic models. Run the app and visit:

- `/docs` for Swagger UI
- `/redoc` for ReDoc
- `/openapi.json` for the raw OpenAPI schema


## Try it

After running `uvicorn main:app --reload --port 8000`, try this in the terminal:

```
curl http://127.0.0.1:8000/
```

Or use Python requests in another script:

```python
import requests
response = requests.get('http://127.0.0.1:8000/')
print(response.json())
```
